# Step 2 — Create a Bare Genie Space on the Gold Tables

**Mistplay Genie Workshop**

You now have a documented gold star schema (Step 1). In this step you create a **Genie space** and attach a few gold tables — nothing else yet. This is your baseline.

You'll see two question sets:
- **Set A** — questions the space can already answer (thanks to the tables + the rich column comments).
- **Set B** — questions it *cannot* answer yet, because they depend on **value matching** (users say "PayPal", the data stores the code `PYPL`). That gap motivates **Step 3**.

> All authoring goes through the **public Genie spaces REST API** (`/api/2.0/genie/spaces`) via the helper in `workshop_lib/genie_builder.py` — so every change in this workshop is reproducible in code.

In [ ]:
import os, sys, importlib
nb_dir = os.getcwd()
if nb_dir not in sys.path:
    sys.path.insert(0, nb_dir)
import workshop_lib.config, workshop_lib.genie_builder, workshop_lib.questions
for m in (workshop_lib.config, workshop_lib.genie_builder, workshop_lib.questions):
    importlib.reload(m)
from workshop_lib.config import WorkshopConfig, setup_widgets
from workshop_lib import genie_builder as gb
from workshop_lib import questions
print("loaded")

In [ ]:
setup_widgets(dbutils)
cfg = WorkshopConfig.from_widgets(dbutils)
assert cfg.warehouse_id, "Set the 'SQL Warehouse ID' widget — Genie needs a warehouse."
print("Config:", cfg.summary())

## 1. Build the space definition

We attach the six gold tables. Genie reads the table & column comments you wrote in Step 1, so even this bare space is reasonably capable. We also seed a few **sample questions** that appear in the Genie UI.

In [ ]:
GOLD_TABLES = ["dim_customers", "dim_products", "dim_geography",
               "fact_orders", "fact_transactions", "fact_shipments"]

builder = gb.SpaceBuilder()
for t in GOLD_TABLES:
    builder.add_table(cfg.gold(t))

builder.add_sample_question("What is the total net revenue across all orders?")
builder.add_sample_question("How many orders were placed through each sales channel?")
builder.add_sample_question("How many customers do we have in each segment?")

print(builder.summary())

## 2. Create the Genie space (REST API)

This calls `POST /api/2.0/genie/spaces`. The space is created under your user folder and named per-attendee so a room of people don't collide. We persist the new **space id** to a small state table so Steps 3-7 can find it.

In [ ]:
client = gb.GenieClient()
parent_path = f"/Workspace/Users/{cfg.user_email}" if cfg.user_email else "/Workspace"

space_id = client.create_space(
    title=cfg.genie_space_name,
    description="Mistplay Genie Workshop — orders, transactions & shipments analytics.",
    parent_path=parent_path,
    warehouse_id=cfg.warehouse_id,
    serialized_space=builder.to_serialized(),
)
gb.save_state(spark, cfg, gb.SPACE_ID_KEY, space_id)

print("Created Genie space:", space_id)
print("Open it here      :", client.space_url(space_id))

## 3. Verify (round-trip)

Fetch the space back and confirm the six tables are attached.

In [ ]:
fetched = client.load_builder(space_id)
attached = [t["identifier"] for t in fetched.to_dict()["data_sources"]["tables"]]
print("Attached tables:")
for t in attached:
    print("  -", t)
assert len(attached) == len(GOLD_TABLES), "Not all tables attached!"
print("\nOK — space has all", len(attached), "gold tables.")

## 4. Try it — the two question sets

Open your Genie space (link above) and ask these. **Set A** should work. **Set B** should struggle — that's the point, and Step 3 fixes it.

In [ ]:
print(questions.render(2))

## Recap & what's next

Your baseline Genie space answers straightforward aggregations because the gold tables are well-commented. But it can't map business language like "PayPal" to the stored code `PYPL`, or "California" to `CA`.

**→ Step 3** adds **value matching** (synonyms + entity matching) so those Set B questions start working.